In [1]:
import sys
print(sys.executable)

/Users/alexisfernandez/Coding/personal/portfolio/project1/.venv/bin/python


In [2]:
import pandas as pd
import numpy as np
import yfinance as yf

In [3]:
#Chosen stocks
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "META",
    "NVDA", "JPM", "XOM", "JNJ", "PG"
]

In [4]:
#Download data
prices = yf.download(tickers, start='2015-01-01', end = '2026-01-01', auto_adjust=True)
prices.head()

[*********************100%***********************]  10 of 10 completed


Price           Close                                                       \
Ticker           AAPL     AMZN      GOOGL        JNJ        JPM       META   
Date                                                                         
2015-01-02  24.171759  15.4260  26.227921  75.737366  46.066780  77.705750   
2015-01-05  23.490801  15.1095  25.728176  75.208412  44.636646  76.457718   
2015-01-06  23.493013  14.7645  25.093222  74.838844  43.479252  75.427582   
2015-01-07  23.822432  14.9210  25.019424  76.490990  43.545597  75.427582   
2015-01-08  24.737753  15.0230  25.106598  77.092438  44.518688  77.438301   

Price                                                  ...     Volume  \
Ticker           MSFT      NVDA         PG        XOM  ...       AAPL   
Date                                                   ...              
2015-01-02  39.607166  0.481884  65.070000  56.777893  ...  212818400   
2015-01-05  39.242962  0.473745  64.760635  55.224358  ...  257142000   
2015-01-06  38.666985  0.459382  64.465630  54.930779  ...  263188400   
2015-01-07  39.158257  0.458185  64.803810  55.487347  ...  160423600   
2015-01-08  40.310211  0.475420  65.544846  56.410946  ...  237458000   

Price                                                                  \
Ticker          AMZN     GOOGL      JNJ       JPM      META      MSFT   
Date                                                                    
2015-01-02  55664000  26480000  5753600  12600000  18177500  27913900   
2015-01-05  55484000  41182000  8079300  20100600  26452200  39673900   
2015-01-06  70380000  54456000  7428000  29074100  27399300  36447900   
2015-01-07  52806000  46918000  7931700  23843200  22045300  29114100   
2015-01-08  61768000  73054000  9916000  16971100  23961000  29645200   

Price                                     
Ticker           NVDA       PG       XOM  
Date                                      
2015-01-02  113680000  7251400  10220400  
2015-01-05  197952000  8626100  18502400  
2015-01-06  197764000  7791200  16670700  
2015-01-07  321808000  5986600  13590700  
2015-01-08  283780000  6823300  15487500  

[5 rows x 50 columns]

In [5]:
close = prices["Close"] #Closing prices (daily)
prices_monthly = close.resample("ME").last() #Month's end closing prices
returns_monthly = prices_monthly.pct_change() #Calculate returns

returns_monthly.head()

Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,PG,XOM
Date,,,,,,,,,,
2015-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-02-28,0.100776,0.072293,0.046656,0.030833,0.126884,0.040311,0.093120,0.153383,0.009966,0.020435
2015-03-31,-0.031372,-0.021202,-0.014095,-0.018633,-0.011423,0.041155,-0.072748,-0.051224,-0.037473,-0.039981
2015-04-30,0.005786,0.133513,-0.010691,-0.013916,0.051179,-0.041961,0.196262,0.060678,-0.021850,0.027882
2015-05-31,0.045339,0.017663,-0.006287,0.016837,0.039836,0.005332,-0.030334,0.001462,-0.014086,-0.016772


In [6]:
#target variable
future_returns = returns_monthly.shift(-1)
future_returns.head()

Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,META,MSFT,NVDA,PG,XOM
Date,,,,,,,,,,
2015-01-31,0.100776,0.072293,0.046656,0.030833,0.126884,0.040311,0.093120,0.153383,0.009966,0.020435
2015-02-28,-0.031372,-0.021202,-0.014095,-0.018633,-0.011423,0.041155,-0.072748,-0.051224,-0.037473,-0.039981
2015-03-31,0.005786,0.133513,-0.010691,-0.013916,0.051179,-0.041961,0.196262,0.060678,-0.021850,0.027882
2015-04-30,0.045339,0.017663,-0.006287,0.016837,0.039836,0.005332,-0.030334,0.001462,-0.014086,-0.016772
2015-05-31,-0.037228,0.011323,-0.009682,-0.026763,0.030101,0.083091,-0.057832,-0.091279,-0.001914,-0.023474


## Feature Engineering

In [7]:
# features_df = future_returns.reset_index().melt(
#     id_vars="Date",
#     var_name="Ticker",
#     value_name="Future Returns"
# )

# features_df.head()


### Momentum

The momentum at time $t$ is given by $$ \frac{P_{t-1}}{P_{t-12}} - 1 $$

measuring how much the stock has increased or decreaed over the past year

In [8]:
momentum = prices_monthly.shift(1) / prices_monthly.shift(12) - 1

In [9]:
momentum_df = momentum.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Momentum"
)

features_df = momentum_df.copy()
features_df.tail()

,Date,Ticker,Momentum
1315,2025-08-31,XOM,-0.028234
1316,2025-09-30,XOM,0.010258
1317,2025-10-31,XOM,0.000400
1318,2025-11-30,XOM,-0.003652
1319,2025-12-31,XOM,0.117174


### Volatility

In [10]:
volatility = prices_monthly.rolling(12).std()

In [11]:
volatility_df = volatility.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Volatility"
)
features_df['Volatility'] = volatility_df['Volatility']
features_df.tail()

,Date,Ticker,Momentum,Volatility
1315,2025-08-31,XOM,-0.028234,4.983386
1316,2025-09-30,XOM,0.010258,4.959671
1317,2025-10-31,XOM,0.000400,5.077395
1318,2025-11-30,XOM,-0.003652,5.302774
1319,2025-12-31,XOM,0.117174,5.991956


### Beta

In [12]:
#Download market data
market = yf.download(
    "^GSPC",
    start="2015-01-01",
    end="2026-01-01",
    auto_adjust=True
)

market_prices = market["Close"].resample("ME").last()
market_returns = market_prices.squeeze().pct_change()

[*********************100%***********************]  1 of 1 completed


In [13]:
returns_monthly = returns_monthly.astype(float)
market_returns = market_returns.astype(float)


In [14]:
#Calculate betas
betas = {}
for ticker in tickers:
    betas[ticker] = returns_monthly[ticker].rolling(12).cov(market_returns) / market_returns.rolling(12).var()

betas

{'AAPL': Date
 2015-01-31         NaN
 2015-02-28         NaN
 2015-03-31         NaN
 2015-04-30         NaN
 2015-05-31         NaN
                 ...   
 2025-08-31    0.368904
 2025-09-30    0.493242
 2025-10-31    0.463810
 2025-11-30    0.389569
 2025-12-31    0.619201
 Freq: ME, Length: 132, dtype: float64,
 'MSFT': Date
 2015-01-31         NaN
 2015-02-28         NaN
 2015-03-31         NaN
 2015-04-30         NaN
 2015-05-31         NaN
                 ...   
 2025-08-31    1.342136
 2025-09-30    1.298322
 2025-10-31    1.202573
 2025-11-30    1.352793
 2025-12-31    1.473588
 Freq: ME, Length: 132, dtype: float64,
 'GOOGL': Date
 2015-01-31         NaN
 2015-02-28         NaN
 2015-03-31         NaN
 2015-04-30         NaN
 2015-05-31         NaN
                 ...   
 2025-08-31    0.879256
 2025-09-30    1.034016
 2025-10-31    1.122435
 2025-11-30    1.435124
 2025-12-31    1.908740
 Freq: ME, Length: 132, dtype: float64,
 'AMZN': Date
 2015-01-31         NaN
 2015-0

In [15]:
betas_predf = pd.DataFrame(betas)
betas_df = betas_predf.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Beta"
)

betas_df

,Date,Ticker,Beta
0,2015-01-31,AAPL,NaN
1,2015-02-28,AAPL,NaN
2,2015-03-31,AAPL,NaN
3,2015-04-30,AAPL,NaN
4,2015-05-31,AAPL,NaN
...,...,...,...
1315,2025-08-31,PG,0.552708
1316,2025-09-30,PG,0.501825
1317,2025-10-31,PG,0.454633
1318,2025-11-30,PG,0.190866


In [16]:
features_df['Beta'] = betas_df['Beta']
features_df.tail()

,Date,Ticker,Momentum,Volatility,Beta
1315,2025-08-31,XOM,-0.028234,4.983386,0.552708
1316,2025-09-30,XOM,0.010258,4.959671,0.501825
1317,2025-10-31,XOM,0.000400,5.077395,0.454633
1318,2025-11-30,XOM,-0.003652,5.302774,0.190866
1319,2025-12-31,XOM,0.117174,5.991956,0.057402


### Current Size \& Value

Note: need to be adjusted into historical data for backtesting, but temporarily uses current data

In [17]:
market_caps = {}
pe_ratios = {}

for ticker in tickers:
    info = yf.Ticker(ticker).info
    market_caps[ticker] = info.get('marketCap')
    pe_ratios[ticker] = info.get('trailingPE')

In [18]:
market_caps

{'AAPL': 4807323025408,
 'MSFT': 3779305668608,
 'GOOGL': 4169429680128,
 'AMZN': 2660659888128,
 'META': 1882072023040,
 'NVDA': 5486440087552,
 'JPM': 890439270400,
 'XOM': 663457038336,
 'JNJ': 644816568320,
 'PG': 344759926784}

In [19]:
#Use log market cap instead for feature
log_market_caps = {
    ticker : float(np.log(cap))
    for ticker, cap in market_caps.items()
    if cap is not None
}

log_market_caps

{'AAPL': 29.20116150154414,
 'MSFT': 28.960561423134347,
 'GOOGL': 29.058800375030355,
 'AMZN': 28.609595286199447,
 'META': 28.263394425774997,
 'NVDA': 29.3333007253417,
 'JPM': 27.514980740158006,
 'XOM': 27.22072993856352,
 'JNJ': 27.19223172309407,
 'PG': 26.566114147357563}

In [20]:
#P/E ratios to Earnings yield

earnings_yield = {
    ticker : 1 / pe
    for ticker, pe in pe_ratios.items()
    if pe is not None and pe > 0
}

earnings_yield

{'AAPL': 0.025804493800986452,
 'MSFT': 0.03528764423251156,
 'GOOGL': 0.05810746766958606,
 'AMZN': 0.05051283152202739,
 'META': 0.037101208386357144,
 'NVDA': 0.03459354743856402,
 'JPM': 0.06928771467152808,
 'XOM': 0.047784317645678136,
 'JNJ': 0.03169263920706538,
 'PG': 0.0444309611287739}

In [21]:
curr_df = pd.DataFrame({
    "Ticker" : list(log_market_caps.keys()),
    "Size" : log_market_caps.values(),
    "Value" : earnings_yield.values()
})

curr_df.head()

,Ticker,Size,Value
0,AAPL,29.201162,0.025804
1,MSFT,28.960561,0.035288
2,GOOGL,29.058800,0.058107
3,AMZN,28.609595,0.050513
4,META,28.263394,0.037101


In [22]:
features_df = features_df.merge(
    curr_df,
    on="Ticker",
    how="left"
)

features_df.tail()

,Date,Ticker,Momentum,Volatility,Beta,Size,Value
1315,2025-08-31,XOM,-0.028234,4.983386,0.552708,27.22073,0.047784
1316,2025-09-30,XOM,0.010258,4.959671,0.501825,27.22073,0.047784
1317,2025-10-31,XOM,0.000400,5.077395,0.454633,27.22073,0.047784
1318,2025-11-30,XOM,-0.003652,5.302774,0.190866,27.22073,0.047784
1319,2025-12-31,XOM,0.117174,5.991956,0.057402,27.22073,0.047784


In [23]:
target_df = future_returns.reset_index().melt(
    id_vars="Date",
    var_name="Ticker",
    value_name="Future Returns"
)

features_df['Future Returns'] = target_df['Future Returns']
features_df.tail()

,Date,Ticker,Momentum,Volatility,Beta,Size,Value,Future Returns
1315,2025-08-31,XOM,-0.028234,4.983386,0.552708,27.22073,0.047784,-0.013474
1316,2025-09-30,XOM,0.010258,4.959671,0.501825,27.22073,0.047784,0.014279
1317,2025-10-31,XOM,0.000400,5.077395,0.454633,27.22073,0.047784,0.022507
1318,2025-11-30,XOM,-0.003652,5.302774,0.190866,27.22073,0.047784,0.038130
1319,2025-12-31,XOM,0.117174,5.991956,0.057402,27.22073,0.047784,NaN
